# 05 - Visualization

This notebook generates publication-style summary figures from model result tables.

## Figure outputs

- `volcano_all.png`, `volcano_male.png`, `volcano_female.png`
- `volcano_sex_interaction.png` (ANCOVA interaction)
- `category_effects_all.png`
- `lipid_distribution_plots/*.png`

## Script equivalent

The same workflow is available via `scripts/05_visualization.py`.


In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from config import DATA_PROCESSED_DIR, FINAL_FORMATTED_FILENAME, FIGURES_DIR, TABLES_DIR, ensure_project_dirs

ensure_project_dirs()


## Step 1: Load result tables and processed data


In [ ]:
df = pd.read_csv(DATA_PROCESSED_DIR / FINAL_FORMATTED_FILENAME)
stats_all = pd.read_csv(TABLES_DIR / "stats_lipid_all.csv")
stats_male = pd.read_csv(TABLES_DIR / "stats_lipid_male.csv")
stats_female = pd.read_csv(TABLES_DIR / "stats_lipid_female.csv")
category_all = pd.read_csv(TABLES_DIR / "stats_category_all.csv")
ancova_lipid = pd.read_csv(TABLES_DIR / "ancova_sex_lipid.csv")


## Step 2: Helper plotting functions


In [ ]:
def make_volcano(stats_df, x_col, p_col, fdr_col, title, x_label, out_path):
    plot_df = stats_df.copy()
    plot_df["neg_log10_p"] = -plot_df[p_col].clip(lower=1e-300).map(math.log10)
    plot_df["significant"] = plot_df[fdr_col] < 0.05

    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        data=plot_df,
        x=x_col,
        y="neg_log10_p",
        hue="significant",
        palette={True: "#d62728", False: "#1f77b4"},
        s=35,
        alpha=0.8,
        linewidth=0,
    )
    plt.axhline(-math.log10(0.05), linestyle="--", color="black", linewidth=1)
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel("-log10(p-value)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.show()


def make_category_barplot(category_df, out_path):
    plot_df = category_df.copy()
    plot_df["category"] = plot_df["lipid"].str.replace("catmean_", "", regex=False)
    plot_df = plot_df.sort_values("coef_primary", ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=plot_df, x="category", y="coef_primary", color="#4c72b0")
    plt.xticks(rotation=45, ha="right")
    plt.title("Category-level SI association (all cohort)")
    plt.xlabel("Lipid category")
    plt.ylabel("SI_avg coefficient")
    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.show()


## Step 3: Generate volcano and category plots


In [ ]:
make_volcano(
    stats_all,
    x_col="coef_primary",
    p_col="p_value_primary",
    fdr_col="fdr_p_value",
    title="Volcano plot: all cohort",
    x_label="SI_avg coefficient",
    out_path=FIGURES_DIR / "volcano_all.png",
)

make_volcano(
    stats_male,
    x_col="coef_primary",
    p_col="p_value_primary",
    fdr_col="fdr_p_value",
    title="Volcano plot: male cohort",
    x_label="SI_avg coefficient",
    out_path=FIGURES_DIR / "volcano_male.png",
)

make_volcano(
    stats_female,
    x_col="coef_primary",
    p_col="p_value_primary",
    fdr_col="fdr_p_value",
    title="Volcano plot: female cohort",
    x_label="SI_avg coefficient",
    out_path=FIGURES_DIR / "volcano_female.png",
)

make_volcano(
    ancova_lipid,
    x_col="coef_interaction",
    p_col="p_interaction",
    fdr_col="fdr_p_interaction",
    title="Volcano plot: ANCOVA sex interaction",
    x_label="SI_avg:msex interaction coefficient",
    out_path=FIGURES_DIR / "volcano_sex_interaction.png",
)

make_category_barplot(category_all, FIGURES_DIR / "category_effects_all.png")


## Step 4: Generate top lipid SI scatter plots


In [ ]:
dist_dir = FIGURES_DIR / "lipid_distribution_plots"
dist_dir.mkdir(parents=True, exist_ok=True)

top = stats_all.sort_values("p_value_primary").head(12)
for _, row in top.iterrows():
    lipid = row["lipid"]
    if lipid not in df.columns:
        continue

    plot_df = df[["SI_avg", lipid]].dropna()
    if plot_df.empty:
        continue

    plt.figure(figsize=(6.5, 5))
    sns.regplot(
        data=plot_df,
        x="SI_avg",
        y=lipid,
        scatter_kws={"alpha": 0.65, "s": 20},
        line_kws={"color": "#d62728", "linewidth": 1.8},
        ci=None,
    )
    plt.title(f"{lipid}\ncoef={row['coef_primary']:.4f}, p={row['p_value_primary']:.2e}")
    plt.xlabel("SI_avg")
    plt.ylabel("Lipid level")
    plt.tight_layout()
    safe_name = lipid.replace("/", "_").replace(" ", "_")
    plt.savefig(dist_dir / f"{safe_name}_vs_SI_avg.png", dpi=220)
    plt.show()

print("Top-level figures:")
for p in sorted(FIGURES_DIR.glob("*.png")):
    print("-", p.name)
print("Top-lipid distribution figures:", len(list(dist_dir.glob("*.png"))))
